[![Ouvrir dans Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/philippe-amuzuga/Meca-IA_Ed2026/blob/main/meca-ia/C19/E19-1_Temperature_Paliers.ipynb)

# Dérive d'un modèle en service : température des paliers d'un parc de pompes

## Le contexte

Dix pompes d'un parc sont relevées chaque jour : charge, vitesse, température de l'air de l'atelier, heures de marche depuis la dernière révision, et température du palier. Un modèle de la température attendue du palier sert de référence de surveillance : une température mesurée très au-dessus de l'attendu alerte l'équipe. Le modèle a été ajusté une fois, enregistré, puis rechargé chaque jour. Mais une pompe s'use : au fil des mois, ses paliers chauffent davantage à charge égale, et le lien appris au début du suivi se déplace. C'est une dérive.

La grandeur à prédire est une température en °C : c'est un problème de régression sur des séries temporelles. La base de test est la fin de chaque pompe, son dernier quart de relevés (1 050 relevés sur 4 200) : elle joue le rôle de la mise en service, quand le modèle prédit l'avenir.

Défi : prédire `temperature_palier_C` et obtenir un R² supérieur à celui du modèle de départ sur la base de test. Ce modèle de départ est une régression linéaire figée sur la charge, la vitesse, l'air ambiant et l'identité de la pompe, apprise sur les trois premiers quarts de chaque pompe ; il atteint un R² de 0,670. Deux comparaisons sans apprentissage : la persistance, qui répond la température de la veille (-0,042), et la moyenne de l'apprentissage (-1,864). À vous de faire mieux. La base de test sert à mesurer, jamais à apprendre ni à régler.

## Les données

Fichier `MecaIA-Temperature_Paliers.xlsx`, dans le dossier `donnees` du dépôt : 4 200 relevés, 7 colonnes. Une ligne est le relevé journalier d'une pompe du parc.

| Colonne | Signification | Unité |
|---|---|---|
| `horodatage` | jour du relevé, 420 relevés par pompe | date |
| `machine` | identifiant de la pompe, `p01` à `p10` | catégorie |
| `charge_pct` | charge moyenne de la journée (quelques cases vides) | % |
| `vitesse_rpm` | vitesse de rotation moyenne de la journée | tr/min |
| `temperature_ambiante_C` | température de l'air de l'atelier (quelques cases vides) | °C |
| `heures_depuis_revision` | heures de marche cumulées depuis la dernière révision | h |
| `temperature_palier_C` | cible : température du palier côté entraînement, en fin de journée | °C |

Origine : jeu simulé, dix pompes à usure de palier quadratique dans les heures de marche et air ambiant saisonnier. Il enseigne une méthode, et aucune conclusion de maintenance n'en découle.

### Étape 1 : charger le fichier et regarder sa forme

On lit le classeur, puis on vérifie ses dimensions, ses cases vides, le pas de temps et le nombre de relevés par pompe : la forme d'un fichier se lit sur le fichier entier. Le dossier des données est cherché depuis le dossier du notebook, puis depuis un clone du dépôt ; ouvert seul, par exemple dans Colab, le notebook télécharge une fois le jeu depuis le dépôt public.

In [ ]:
from pathlib import Path
import urllib.request

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn import set_config

set_config(display="text")
GRAINE = 42
FICHIER = "MecaIA-Temperature_Paliers.xlsx"
URL_JEU = "https://raw.githubusercontent.com/philippe-amuzuga/Meca-IA_Ed2026/main/donnees/MecaIA-Temperature_Paliers.xlsx"

# Le dossier `donnees` du dépôt, cherché depuis le dossier du notebook, puis depuis un clone.
candidats = [Path("..", "..", "donnees"), Path("Meca-IA_Ed2026", "donnees")]
dossier = next((d for d in candidats if (d / FICHIER).is_file()), None)
if dossier is None:
    # Notebook ouvert seul (Colab) : copie unique du jeu depuis le dépôt public.
    dossier = Path("donnees")
    dossier.mkdir(exist_ok=True)
    try:
        urllib.request.urlretrieve(URL_JEU, dossier / FICHIER)
    except OSError as erreur:
        raise FileNotFoundError(
            "Jeu introuvable : clonez le dépôt, ou vérifiez la connexion pour le télécharger.") from erreur
df = pd.read_excel(dossier / FICHIER)

cible = "temperature_palier_C"
vides = df.isna().sum()
print("dimensions :", df.shape)
print("cases vides :", vides[vides > 0].to_dict())
pas = df.groupby("machine")["horodatage"].diff().dropna()
print("pas de temps :", {str(k): int(n) for k, n in pas.value_counts().items()})
print("relevés par pompe :", sorted(df["machine"].value_counts().unique().tolist()))
print("période :", df["horodatage"].min().date(), "au", df["horodatage"].max().date())
print(df[cible].describe().round(1).to_dict())
df.head(3)

Lecture du résultat : 4 200 relevés de 10 pompes au pas d'un jour, du 06/01/2025 au 01/03/2026. Deux colonnes ont des cases vides : la charge (136) et l'air ambiant (91). La température du palier va de 30 à 66 °C. Les relevés des pompes se suivent jour après jour : la base de test ne peut pas être tirée au hasard.

### Préparation du dépôt (Colab)

Ouvert seul dans Colab, ce notebook ne voit pas le dossier `donnees/` du dépôt. La cellule suivante ne télécharge rien : elle cherche ce dossier (à deux dossiers au-dessus du notebook, à côté de lui ou dans un clone nommé `Meca-IA_Ed2026`), vérifie la présence du classeur du défi et, sur Colab, liste ce qui manque. Pour le fournir, ouvrez le panneau des fichiers de Colab, créez le dossier `donnees` à la racine de l'espace de travail et téléversez-y le classeur listé, pris dans le dépôt.

In [ ]:
# Préparation du dépôt (utile sur Colab) : cette cellule ne télécharge rien.
import os
from pathlib import Path

JEUX_LUS = ["MecaIA-Temperature_Paliers.xlsx"]                  # le classeur lu par ce notebook


def chercher(dossier):
    """Le dossier du dépôt cherché depuis le notebook, puis depuis un clone ; None s'il est introuvable."""
    for base in (Path("..", ".."), Path("."), Path("Meca-IA_Ed2026")):
        if (base / dossier).is_dir():
            return base / dossier
    return None


donnees = chercher("donnees")
manque = [f"donnees/{nom}" for nom in JEUX_LUS if donnees is None or not (donnees / nom).is_file()]
if manque and "COLAB_RELEASE_TAG" in os.environ:          # variable posée par Colab
    print("À téléverser dans l'espace de travail de Colab, panneau des fichiers :")
    print("\n".join("  " + nom for nom in manque))


### Étape 2 : ranger le temps et mettre le test de côté

On trie les relevés de chaque pompe par date. Le dernier quart de chaque pompe est mis de côté avant tout apprentissage : il joue le rôle de l'avenir en service. Aucun mélange des lignes n'est permis, car le futur d'une pompe ne doit jamais servir à prédire son passé.

In [ ]:
import math

df = df.sort_values(["machine", "horodatage"])
par_machine = df.groupby("machine")
effectif = par_machine["horodatage"].transform("size")
rang_depuis_la_fin = par_machine.cumcount(ascending=False)           # 0 : dernier relevé de la pompe
est_test = rang_depuis_la_fin < np.ceil(0.25 * effectif)
X = df[["machine", "charge_pct", "vitesse_rpm", "temperature_ambiante_C", "heures_depuis_revision"]]
y = df[cible]
X_train, X_test, y_train, y_test = X[~est_test], X[est_test], y[~est_test], y[est_test]
print("apprentissage :", len(X_train), "relevés, jusqu'au", df.loc[X_train.index, "horodatage"].max().date())
print("test :", len(X_test), "relevés, à partir du", df.loc[X_test.index, "horodatage"].min().date())

### Étape 3 : regarder la dérive avant de mesurer, sur l'apprentissage seul

Un modèle en service ne connaît pas la réponse de demain, mais il voit déjà ses entrées. L'indice de stabilité de population (PSI) compare la répartition d'une entrée sur une période de référence, ici les 60 % les plus anciens relevés de l'apprentissage de chaque pompe, à celle d'une période récente, ses 15 % les plus récents. On le lit ainsi : moins de 0,1, stable ; de 0,1 à 0,25, à surveiller ; au-delà, un changement net. Aucune température de palier n'intervient : le PSI ne regarde que les entrées.

In [ ]:
def psi(reference, recent, classes=10):
    """Indice de stabilité de population entre deux séries, sur les déciles de la référence."""
    bornes = np.unique(np.quantile(reference.dropna(), np.linspace(0, 1, classes + 1)))
    bornes[0], bornes[-1] = -np.inf, np.inf
    p = np.histogram(reference.dropna(), bornes)[0] / reference.notna().sum()
    q = np.histogram(recent.dropna(), bornes)[0] / recent.notna().sum()
    p, q = np.clip(p, 1e-4, None), np.clip(q, 1e-4, None)
    return float(np.sum((q - p) * np.log(q / p)))


avancement = X_train.groupby("machine").cumcount() / X_train.groupby("machine")["charge_pct"].transform("size")
ancien, recent = X_train[avancement < 0.60], X_train[avancement >= 0.85]
entrees_numeriques = ["charge_pct", "vitesse_rpm", "temperature_ambiante_C", "heures_depuis_revision"]
indices_psi = {col: psi(ancien[col], recent[col]) for col in entrees_numeriques}
print({col: round(valeur, 2) for col, valeur in indices_psi.items()})

Lecture du résultat : la charge (0,02) et la vitesse (0,04) sont stables. Les heures de marche (8,3) changent forcément, puisqu'elles croissent à chaque jour. L'air ambiant (3,23) change aussi, par la saison. Un PSI élevé signale qu'une entrée a changé, pas que le modèle se trompe : l'air ambiant a un effet que le modèle a appris, la saison n'est donc pas une dérive du modèle. Seule la comparaison des prédictions aux mesures dira si le lien entre les entrées et la température s'est déplacé.

### Étape 4 : construire le modèle de départ, l'enregistrer et le recharger

Le modèle de départ est une régression linéaire figée : l'identité de la pompe devient des colonnes de 0 et de 1, les cases vides de la charge et de l'air sont remplacées par la médiane de l'apprentissage, les trois mesures du jour sont mises à l'échelle, et les heures de marche ne servent pas. Le pipeline apprend sur `X_train` seul. Un modèle en service est ensuite enregistré, puis rechargé : `joblib` sérialise le pipeline entier, préparation comprise, avec la liste de ses colonnes et la date de la dernière donnée apprise. Ici l'enregistrement se fait en mémoire ; sur un poste, on écrirait un fichier `.joblib`. Rechargé, le modèle doit répondre exactement comme avant.

In [ ]:
# Votre code :

In [ ]:
# Votre code :

### Étape 5 : mesurer sur la base de test

Le modèle rechargé prédit des journées qu'il n'a jamais vues, les plus récentes de chaque pompe. Trois indicateurs sont lus : le coefficient de détermination R², et deux erreurs en °C, la racine de l'erreur quadratique moyenne (RMSE) et l'erreur absolue moyenne (MAE), avec le biais moyen, qui dit si le modèle sous-estime. Les deux comparaisons sans apprentissage donnent le point de comparaison.

In [ ]:
from sklearn.metrics import mean_absolute_error, r2_score, root_mean_squared_error

y_pred = recharge["modele"].predict(X_test[recharge["colonnes"]])
score_depart = r2_score(y_test, y_pred)
veille = par_machine[cible].shift(1)[est_test].fillna(y_train.mean())
score_persistance = r2_score(y_test, veille)
score_temoin = r2_score(y_test, np.full(len(y_test), y_train.mean()))
biais = (y_test - y_pred).mean()
print(f"R² : {score_depart:.3f}")
print(f"RMSE : {root_mean_squared_error(y_test, y_pred):.2f} °C")
print(f"MAE : {mean_absolute_error(y_test, y_pred):.2f} °C")
print(f"biais moyen (mesuré moins prédit) : {biais:+.2f} °C")
print(f"R² de la persistance (température de la veille) : {score_persistance:.3f}")
print(f"R² du témoin (toujours la moyenne de l'apprentissage) : {score_temoin:.3f}")

Lecture du résultat : le R² est de 0,670, l'erreur absolue moyenne de 2,03 °C et le biais de 1,64 °C : le modèle sous-estime en moyenne, parce que les paliers de la période de test chauffent plus à charge égale que ceux de la période d'apprentissage. La persistance, qui ne sait rien, atteint -0,042 et la moyenne -1,864 : la moyenne de l'apprentissage est un mauvais repère, car la saison a changé. C'est ce chiffre, mesuré sur des journées que le modèle n'a jamais vues, que l'on a le droit d'annoncer.

## Pour aller plus loin : suivre la dérive, puis réentraîner

Le modèle de départ de l'étape 4 reste le score de référence du défi. Les cellules suivantes montrent comment le dépasser sans regarder le test pour choisir. La première rejoue le service sur l'apprentissage : on ajuste le modèle de départ sur les trois premiers quarts de chaque pompe, puis on regarde l'erreur moyenne par tranches de 15 jours sur le dernier quart, qui joue ici le rôle de période de service. La deuxième compare quatre modèles sur cette même validation chronologique : la droite figée, une droite qui lit aussi les heures de marche et leur carré, la droite figée réentraînée sur les 90 derniers jours de l'ajustement seulement, et une forêt aléatoire qui lit les heures de marche. La dernière mesure le modèle retenu, ajusté sur tout l'apprentissage, une seule fois sur le test.

In [ ]:
# Votre code :

In [ ]:
# Votre code :

In [ ]:
# Votre code :

Lecture du résultat : rejoué sur l'apprentissage, le modèle de départ sous-estime de plus en plus : l'erreur moyenne passe de 0,62 °C sur la première tranche de service à 1,60 °C sur la dernière. C'est la signature d'une dérive, et elle se voit avant d'avoir un seul résultat de test. Sur la validation chronologique, la droite figée atteint 0,916, la droite réentraînée sur 90 jours 0,935, la droite qui lit les heures de marche et leur carré 0,899 et la forêt 0,755 : le modèle retenu est « droite figée réentraînée sur 90 jours ». Mesuré une seule fois sur le test, il donne un R² de 0,860, soit 0,190 de plus que le modèle de départ, et un biais de 0,21 °C. La forêt, qui coupe l'espace des entrées, ne prolonge pas la tendance des heures de marche : elle est la moins bonne. La droite qui lit les heures de marche fait moins bien que la droite figée sur cette validation, sans doute parce que l'usure y est encore faible : un modèle de la cause a besoin d'avoir vu la cause varier. Réentraîner sur une fenêtre récente suit la dérive sans la modéliser.

## Le score de référence

Le score publié est le R² du modèle de départ de l'étape 4, rechargé, sur la base de test. L'empreinte identifie cette base de test : si la vôtre diffère, le découpage n'est pas le même et les scores ne se comparent pas.

In [ ]:
import hashlib

score = r2_score(y_test, recharge["modele"].predict(X_test[recharge["colonnes"]]))
positions = ",".join(str(i) for i in sorted(X_test.index))
print("EMPREINTE_TEST =", hashlib.sha256(positions.encode("ascii")).hexdigest())
print("SCORE_REFERENCE r2 = %.3f" % score)

## Synthèse

1. Un modèle en service s'enregistre avec sa préparation (`joblib`), la liste de ses colonnes et la date de sa dernière donnée apprise ; rechargé, il répond exactement comme avant.
2. Le PSI compare les entrées d'une période récente à celles d'une période de référence : il détecte un changement des entrées (la saison, les heures de marche), pas une dégradation du modèle.
3. Une dérive du lien entre les entrées et la cible se voit sur l'erreur moyenne par tranches de service : de 0,62 à 1,60 °C ici, avant même le test.
4. Réentraîner sur une fenêtre récente suit la dérive sans la modéliser : 0,935 contre 0,916 sur la validation chronologique. Un modèle qui lit la cause (les heures de marche) n'aide que s'il a vu la cause varier ; un modèle qui coupe les entrées ne prolonge aucune tendance.
5. Mesuré une seule fois sur le test, le modèle retenu donne 0,860, contre 0,670 pour le modèle figé.